<a href="https://colab.research.google.com/github/poggerssLL/Automatica---Grupo-2/blob/main/etapa-02-grafos/17_Problemas_Hamiltonianos_e_Roteamento_de_AGV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 17 - Notebook: Problema do Caixeiro‑Viajante (TSP) e Roteamento Logístico de AGVs na Máquina Envasadora de Copos

Neste notebook implementamos a heurística **Nearest Neighbor** e a busca local **2‑Opt** para otimizar a rota de um AGV que deve visitar diversas estações da linha de envase (controle de qualidade, medição de peso, inspeção visual etc.) e retornar ao ponto de partida.

In [1]:
from typing import List

class RoteadorEnvasadora_TSP:
    def __init__(self, locais: List[str], matriz_distancias: List[List[float]]):
        self.locais = locais
        self.dist = matriz_distancias
        self.n = len(locais)

    def calcular_custo_rota(self, rota: List[int]) -> float:
        # soma das distâncias ao longo da rota + retorno ao início
        custo = sum(self.dist[rota[i]][rota[i+1]] for i in range(len(rota)-1))
        custo += self.dist[rota[-1]][rota[0]]
        return custo

    def vizinho_mais_proximo(self, inicio_idx: int = 0) -> List[int]:
        nao_vis = set(range(self.n))
        nao_vis.remove(inicio_idx)
        rota = [inicio_idx]
        atual = inicio_idx
        while nao_vis:
            prox = min(nao_vis, key=lambda v: self.dist[atual][v])
            rota.append(prox)
            nao_vis.remove(prox)
            atual = prox
        return rota

    def otimizar_2opt(self, rota: List[int]) -> List[int]:
        melhor = list(rota)
        melhor_c = self.calcular_custo_rota(melhor)
        melhorou = True
        while melhorou:
            melhorou = False
            for i in range(1, self.n - 1):
                for j in range(i + 1, self.n):
                    nova = melhor[:i] + melhor[i:j+1][::-1] + melhor[j+1:]
                    c = self.calcular_custo_rota(nova)
                    if c < melhor_c:
                        melhor, melhor_c = nova, c
                        melhorou = True
                        break
                if melhorou:
                    break
        return melhor

# Dados de exemplo (estações da linha de envase)
locais = ["Lab", "TK301", "TK302", "R101", "TK303", "GRAN201"]
dist_mat = [
    [0.0, 45.0, 50.0, 80.0, 75.0, 110.0],
    [45.0, 0.0, 20.0, 55.0, 60.0, 95.0],
    [50.0, 20.0, 0.0, 45.0, 50.0, 85.0],
    [80.0, 55.0, 45.0, 0.0, 25.0, 40.0],
    [75.0, 60.0, 50.0, 25.0, 0.0, 35.0],
    [110.0, 95.0, 85.0, 40.0, 35.0, 0.0]
]

agv = RoteadorEnvasadora_TSP(locais, dist_mat)
rota_ini = agv.vizinho_mais_proximo(0)
rota_opt = agv.otimizar_2opt(rota_ini)

print(f"Rota Inicial NN: {' -> '.join(locais[i] for i in rota_ini)} | Custo: {agv.calcular_custo_rota(rota_ini):.1f}s")
print(f"Rota Ótima 2‑Opt: {' -> '.join(locais[i] for i in rota_opt)} | Custo: {agv.calcular_custo_rota(rota_opt):.1f}s")


Rota Inicial NN: Lab -> TK301 -> TK302 -> R101 -> TK303 -> GRAN201 | Custo: 280.0s
Rota Ótima 2‑Opt: Lab -> TK301 -> TK302 -> R101 -> GRAN201 -> TK303 | Custo: 260.0s
